In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [1]:
!pip install -q spacy

In [2]:
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 96.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [3]:
from pathlib import Path
import re
import json

import pandas as pd
import spacy

In [4]:
# Load spaCy model

nlp = spacy.load("en_core_web_sm")

print("spaCy model loaded successfully.")

spaCy model loaded successfully.


In [5]:
PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Colab Notebooks/Intelligent_Document_Processing"
)

CLEANED_PATH = PROJECT_ROOT / "outputs" / "cleaned"

NER_OUTPUT_PATH = PROJECT_ROOT / "outputs" / "ner"

In [7]:
for folder in [
    NER_OUTPUT_PATH / "invoices",
    NER_OUTPUT_PATH / "resumes",
    NER_OUTPUT_PATH / "id_cards"
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )

print("NER output folders ready.")

NER output folders ready.


In [8]:
# Verify cleaned data

print("Invoices:",
      (CLEANED_PATH / "invoices").exists())

print("Resumes:",
      (CLEANED_PATH / "resumes").exists())

print("ID Cards:",
      (CLEANED_PATH / "id_cards").exists())

Invoices: True
Resumes: True
ID Cards: True


In [9]:
# Find cleaned text files

def get_text_files(folder):

    folder = Path(folder)

    return sorted(
        file
        for file in folder.glob("*.txt")
        if file.is_file()
    )

In [10]:
invoice_files = get_text_files(
    CLEANED_PATH / "invoices"
)

resume_files = get_text_files(
    CLEANED_PATH / "resumes"
)

id_card_files = get_text_files(
    CLEANED_PATH / "id_cards"
)

print("Invoice files:", len(invoice_files))
print("Resume files :", len(resume_files))
print("ID Card files:", len(id_card_files))

Invoice files: 3
Resume files : 2
ID Card files: 3


In [11]:
def read_text_file(file_path):

    with open(
        file_path,
        "r",
        encoding="utf-8"
    ) as file:

        return file.read()

In [12]:
sample_file = invoice_files[0]

sample_text = read_text_file(
    sample_file
)

print(sample_text)

tan woon yann
BOOK TA _K (TAMAN DAYA) SDN BHD
789H17-W
NO.S: 55,57 & S9, JALAN SAGU I8,
TAMAN DAYA,
81100 JOIIOR BAHRU,
JOHOR
HHE
Dociinent No
Dale
25/12p2018 8: 13.39 PM
Cashier
MANIS
Menxler
CASH BILL
CODE/DESc
Disc
Abl( J J!l
QTY
Ri
RM
1 PC
9.00)
0,C0
9.00
Total
S
Rour: ling Acljustnienl;
0.00
Round: d Total (RM): Cash
CHIANGE
0o
EXCH INNIGEABLE
# #ii i &
THANK YOU
PLEASE COI 'E AGATN


In [13]:
# General spaCy NER

def extract_spacy_entities(text):

    doc = nlp(text)

    entities = []

    for ent in doc.ents:

        entities.append({
            "text": ent.text,
            "label": ent.label_,
            "start": ent.start_char,
            "end": ent.end_char
        })

    return entities

In [14]:
entities = extract_spacy_entities(
    sample_text
)

entities

[{'text': '55,57 & S9', 'label': 'ORG', 'start': 61, 'end': 71},
 {'text': 'SAGU I8', 'label': 'ORG', 'start': 79, 'end': 86},
 {'text': 'TAMAN DAYA', 'label': 'PERSON', 'start': 88, 'end': 98},
 {'text': '81100', 'label': 'CARDINAL', 'start': 100, 'end': 105},
 {'text': '25/12p2018', 'label': 'CARDINAL', 'start': 148, 'end': 158},
 {'text': '8', 'label': 'CARDINAL', 'start': 159, 'end': 160},
 {'text': '13.39', 'label': 'CARDINAL', 'start': 162, 'end': 167},
 {'text': 'Cashier', 'label': 'ORG', 'start': 171, 'end': 178},
 {'text': '1', 'label': 'CARDINAL', 'start': 239, 'end': 240},
 {'text': '9.00', 'label': 'CARDINAL', 'start': 244, 'end': 248},
 {'text': '0,C0', 'label': 'CARDINAL', 'start': 250, 'end': 254},
 {'text': '9.00', 'label': 'CARDINAL', 'start': 255, 'end': 259},
 {'text': 'Acljustnienl', 'label': 'PERSON', 'start': 279, 'end': 291},
 {'text': '0.00', 'label': 'CARDINAL', 'start': 293, 'end': 297},
 {'text': '0o', 'label': 'CARDINAL', 'start': 332, 'end': 334},
 {'text':

In [15]:
# Display entities

for entity in entities:

    print(
        f"{entity['text']} "
        f"--> "
        f"{entity['label']}"
    )

55,57 & S9 --> ORG
SAGU I8 --> ORG
TAMAN DAYA --> PERSON
81100 --> CARDINAL
25/12p2018 --> CARDINAL
8 --> CARDINAL
13.39 --> CARDINAL
Cashier --> ORG
1 --> CARDINAL
9.00 --> CARDINAL
0,C0 --> CARDINAL
9.00 --> CARDINAL
Acljustnienl --> PERSON
0.00 --> CARDINAL
0o --> CARDINAL
# #ii --> MONEY


In [16]:
# Extract Email

def extract_emails(text):

    pattern = r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b'

    emails = re.findall(
        pattern,
        text
    )

    return list(dict.fromkeys(
        email.lower()
        for email in emails
    ))

In [17]:
extract_emails(sample_text)

[]

In [18]:
def extract_phone_numbers(text):

    pattern = (
        r'(?<!\d)'
        r'(?:\+91[\s-]?)?'
        r'[6-9]\d{9}'
        r'(?!\d)'
    )

    phones = re.findall(
        pattern,
        text
    )

    return list(dict.fromkeys(
        phone.strip()
        for phone in phones
    ))

In [19]:
extract_phone_numbers(sample_text)

[]

In [20]:
def extract_dates(text):

    pattern = (
        r'\b'
        r'(?:'
        r'\d{1,2}[/-]\d{1,2}[/-]\d{2,4}'
        r'|'
        r'\d{1,2}[/-][A-Za-z]{3,9}[/-]?\d{2,4}'
        r'|'
        r'[A-Za-z]{3,9}\s+\d{1,2},?\s+\d{2,4}'
        r')'
        r'\b'
    )

    dates = re.findall(
        pattern,
        text,
        flags=re.IGNORECASE
    )

    return list(dict.fromkeys(
        date.strip()
        for date in dates
    ))

In [21]:
extract_dates(sample_text)

[]

In [22]:
def extract_amounts(text):

    pattern = (
        r'(?:₹|Rs\.?|INR|\$|€|£)\s*'
        r'\d+(?:,\d{3})*(?:\.\d{1,2})?'
        r'|'
        r'\b\d+(?:,\d{3})+(?:\.\d{1,2})?\b'
    )

    amounts = re.findall(
        pattern,
        text,
        flags=re.IGNORECASE
    )

    return list(dict.fromkeys(
        amount.strip()
        for amount in amounts
    ))

In [23]:
extract_amounts(sample_text)

[]

In [24]:
def extract_invoice_number(text):

    patterns = [
        r'(?:invoice\s*(?:no|number|#)\s*[:\-]?\s*)([A-Z0-9\/\-]+)',
        r'(?:inv\s*(?:no|#)\s*[:\-]?\s*)([A-Z0-9\/\-]+)'
    ]

    for pattern in patterns:

        match = re.search(
            pattern,
            text,
            flags=re.IGNORECASE
        )

        if match:
            return match.group(1).strip()

    return None

In [25]:
extract_invoice_number(sample_text)

In [26]:
def extract_gstin(text):

    pattern = (
        r'\b'
        r'\d{2}[A-Z]{5}\d{4}[A-Z]'
        r'\d[A-Z0-9]'
        r'\b'
    )

    matches = re.findall(
        pattern,
        text.upper()
    )

    return list(dict.fromkeys(matches))

In [27]:
extract_gstin(sample_text)

[]

In [28]:
def extract_aadhaar_like(text):

    pattern = r'(?<!\d)\d{4}\s\d{4}\s\d{4}(?!\d)'

    matches = re.findall(
        pattern,
        text
    )

    return list(dict.fromkeys(matches))

In [29]:
def extract_spacy_by_label(text):

    doc = nlp(text)

    entities = {}

    for ent in doc.ents:

        label = ent.label_

        if label not in entities:
            entities[label] = []

        if ent.text not in entities[label]:

            entities[label].append(
                ent.text
            )

    return entities

In [30]:
extract_spacy_by_label(sample_text)

{'ORG': ['55,57 & S9', 'SAGU I8', 'Cashier'],
 'PERSON': ['TAMAN DAYA', 'Acljustnienl'],
 'CARDINAL': ['81100',
  '25/12p2018',
  '8',
  '13.39',
  '1',
  '9.00',
  '0,C0',
  '0.00',
  '0o'],
 'MONEY': ['# #ii']}

In [31]:
def extract_entities(
    text,
    document_type
):

    result = {
        "emails": extract_emails(text),
        "phone_numbers": extract_phone_numbers(text),
        "dates": extract_dates(text),
        "amounts": extract_amounts(text),
        "spacy_entities":
            extract_spacy_by_label(text)
    }

    if document_type == "invoice":

        result["invoice_number"] = (
            extract_invoice_number(text)
        )

        result["gstin"] = (
            extract_gstin(text)
        )

    elif document_type == "resume":

        result["persons"] = (
            result["spacy_entities"]
            .get("PERSON", [])
        )

        result["organizations"] = (
            result["spacy_entities"]
            .get("ORG", [])
        )

        result["locations"] = (
            result["spacy_entities"]
            .get("GPE", [])
        )

    elif document_type == "id_card":

        result["persons"] = (
            result["spacy_entities"]
            .get("PERSON", [])
        )

        result["id_numbers"] = (
            extract_aadhaar_like(text)
        )

    return result

In [32]:
# Test Invoice Extraction

invoice_entities = extract_entities(
    sample_text,
    "invoice"
)

invoice_entities

{'emails': [],
 'phone_numbers': [],
 'dates': [],
 'amounts': [],
 'spacy_entities': {'ORG': ['55,57 & S9', 'SAGU I8', 'Cashier'],
  'PERSON': ['TAMAN DAYA', 'Acljustnienl'],
  'CARDINAL': ['81100',
   '25/12p2018',
   '8',
   '13.39',
   '1',
   '9.00',
   '0,C0',
   '0.00',
   '0o'],
  'MONEY': ['# #ii']},
 'invoice_number': None,
 'gstin': []}

In [33]:
print(
    json.dumps(
        invoice_entities,
        indent=4,
        ensure_ascii=False
    )
)

{
    "emails": [],
    "phone_numbers": [],
    "dates": [],
    "amounts": [],
    "spacy_entities": {
        "ORG": [
            "55,57 & S9",
            "SAGU I8",
            "Cashier"
        ],
        "PERSON": [
            "TAMAN DAYA",
            "Acljustnienl"
        ],
        "CARDINAL": [
            "81100",
            "25/12p2018",
            "8",
            "13.39",
            "1",
            "9.00",
            "0,C0",
            "0.00",
            "0o"
        ],
        "MONEY": [
            "# #ii"
        ]
    },
    "invoice_number": null,
    "gstin": []
}


In [34]:
# Test Resume

sample_resume = resume_files[0]

resume_text = read_text_file(
    sample_resume
)

resume_entities = extract_entities(
    resume_text,
    "resume"
)

print(
    json.dumps(
        resume_entities,
        indent=4,
        ensure_ascii=False
    )
)

{
    "emails": [],
    "phone_numbers": [],
    "dates": [],
    "amounts": [],
    "spacy_entities": {
        "PERSON": [
            "Diana Dawa",
            "Mukaa"
        ],
        "CARDINAL": [
            "4"
        ],
        "ORG": [
            "Data Sdendtt\nTECHNICAL"
        ]
    },
    "persons": [
        "Diana Dawa",
        "Mukaa"
    ],
    "organizations": [
        "Data Sdendtt\nTECHNICAL"
    ],
    "locations": []
}


In [35]:
# Test ID Card

sample_id = id_card_files[0]

id_text = read_text_file(
    sample_id
)

id_entities = extract_entities(
    id_text,
    "id_card"
)

print(
    json.dumps(
        id_entities,
        indent=4,
        ensure_ascii=False
    )
)

{
    "emails": [],
    "phone_numbers": [],
    "dates": [
        "31/10/1992"
    ],
    "amounts": [],
    "spacy_entities": {
        "ORG": [
            "NNCOHE TAXDEPAMHENT\n",
            "GOVT OF NNDIA"
        ],
        "PERSON": [
            "MAHADEV SHINDE",
            "Accounl Numbcr"
        ]
    },
    "persons": [
        "MAHADEV SHINDE",
        "Accounl Numbcr"
    ],
    "id_numbers": []
}


In [36]:
# Batch NER Function

def batch_ner(
    input_folder,
    document_type,
    max_files=3
):

    files = get_text_files(
        input_folder
    )

    if max_files is not None:
        files = files[:max_files]

    results = {}

    for file_path in files:

        try:

            text = read_text_file(
                file_path
            )

            entities = extract_entities(
                text,
                document_type
            )

            results[file_path.name] = {
                "document_type":
                    document_type,
                "entities":
                    entities
            }

        except Exception as e:

            print(
                f"Error processing "
                f"{file_path.name}: {e}"
            )

    return results

In [37]:
# Run NER on 3 Invoices

invoice_ner = batch_ner(
    CLEANED_PATH / "invoices",
    "invoice",
    max_files=3
)

print(
    "Invoices processed:",
    len(invoice_ner)
)

Invoices processed: 3


In [38]:
# Run NER on 3 Resumes

resume_ner = batch_ner(
    CLEANED_PATH / "resumes",
    "resume",
    max_files=3
)

print(
    "Resumes processed:",
    len(resume_ner)
)

Resumes processed: 2


In [39]:
# Run NER on 3 ID Cards

id_card_ner = batch_ner(
    CLEANED_PATH / "id_cards",
    "id_card",
    max_files=3
)

print(
    "ID cards processed:",
    len(id_card_ner)
)

ID cards processed: 3


In [40]:
# Save JSON

def save_json(
    data,
    output_file
):

    output_file = Path(
        output_file
    )

    output_file.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    with open(
        output_file,
        "w",
        encoding="utf-8"
    ) as file:

        json.dump(
            data,
            file,
            indent=4,
            ensure_ascii=False
        )

    print(
        f"Saved: {output_file}"
    )

In [41]:
save_json(
    invoice_ner,
    NER_OUTPUT_PATH /
    "invoice_entities.json"
)

save_json(
    resume_ner,
    NER_OUTPUT_PATH /
    "resume_entities.json"
)

save_json(
    id_card_ner,
    NER_OUTPUT_PATH /
    "id_card_entities.json"
)

Saved: /content/drive/MyDrive/Colab Notebooks/Intelligent_Document_Processing/outputs/ner/invoice_entities.json
Saved: /content/drive/MyDrive/Colab Notebooks/Intelligent_Document_Processing/outputs/ner/resume_entities.json
Saved: /content/drive/MyDrive/Colab Notebooks/Intelligent_Document_Processing/outputs/ner/id_card_entities.json


In [42]:
def ner_to_dataframe(
    results
):

    rows = []

    for filename, data in results.items():

        entities = data["entities"]

        rows.append({
            "filename": filename,
            "document_type":
                data["document_type"],

            "emails":
                ", ".join(
                    entities.get(
                        "emails", []
                    )
                ),

            "phone_numbers":
                ", ".join(
                    entities.get(
                        "phone_numbers", []
                    )
                ),

            "dates":
                ", ".join(
                    entities.get(
                        "dates", []
                    )
                ),

            "amounts":
                ", ".join(
                    entities.get(
                        "amounts", []
                    )
                ),

            "invoice_number":
                entities.get(
                    "invoice_number"
                ),

            "gstin":
                ", ".join(
                    entities.get(
                        "gstin", []
                    )
                ),

            "persons":
                ", ".join(
                    entities.get(
                        "persons", []
                    )
                ),

            "organizations":
                ", ".join(
                    entities.get(
                        "organizations", []
                    )
                ),

            "locations":
                ", ".join(
                    entities.get(
                        "locations", []
                    )
                ),

            "id_numbers":
                ", ".join(
                    entities.get(
                        "id_numbers", []
                    )
                )
        })

    return pd.DataFrame(rows)

In [43]:
invoice_df = ner_to_dataframe(
    invoice_ner
)

resume_df = ner_to_dataframe(
    resume_ner
)

id_card_df = ner_to_dataframe(
    id_card_ner
)

In [44]:
invoice_df

,filename,document_type,emails,phone_numbers,dates,amounts,invoice_number,gstin,persons,organizations,locations,id_numbers
0,X00016469612.txt,invoice,,,,,None,,,,,
1,X00016469620.txt,invoice,,,12/01/19,,None,,,,,
2,X00016469622.txt,invoice,,,,,None,,,,,


In [45]:
invoice_df.to_csv(
    NER_OUTPUT_PATH /
    "invoice_entities.csv",
    index=False
)

resume_df.to_csv(
    NER_OUTPUT_PATH /
    "resume_entities.csv",
    index=False
)

id_card_df.to_csv(
    NER_OUTPUT_PATH /
    "id_card_entities.csv",
    index=False
)

print("CSV files saved successfully.")

CSV files saved successfully.
